# Selecting Data

Selecting means: **"give me only these rows and these columns."** It is the most used skill in pandas. Every cleaning step, every feature, every train/test split starts with a selection.

**Contents**
1. The selection toolbox
2. Our dataset
3. Selecting columns
4. Selecting rows: the `[]` slicing quirk
5. `loc` (by label)
6. `iloc` (by position)
7. `loc` vs `iloc` (and the index trap)
8. Boolean filtering
9. `isin`, `between`, string filters, missing-value filters
10. `query()`
11. `at`, `iat`, `where`, `np.where`
12. Setting values safely (and Copy-on-Write)
13. Selecting with a custom index
14. Mini AI/ML example: X, y and train/test split
15. Pandas 3.x notes
16. Common mistakes
17. Interview questions
18. Practice questions with solutions
19. Cheat sheet

In [2]:
import numpy as np
import pandas as pd
import warnings

print("pandas:", pd.__version__)

pandas: 3.0.6


---
# 1. The selection toolbox

| I want to... | Use |
|---|---|
| One column (as Series) | `df["col"]` |
| Several columns (as DataFrame) | `df[["a", "b"]]` |
| Rows/columns by **label** | `df.loc[rows, cols]` |
| Rows/columns by **position** | `df.iloc[rows, cols]` |
| Rows that satisfy a condition | `df[condition]` or `df.loc[condition]` |
| Value is in a list | `df[df["col"].isin([...])]` |
| Value in a range | `df[df["col"].between(a, b)]` |
| Text contains something | `df[df["col"].str.contains("x")]` |
| Condition written as a string | `df.query("score > 80")` |
| Single value, fast | `df.at[label, col]`, `df.iat[row, col]` |
| Columns by dtype | `df.select_dtypes(...)` |

If you remember only one thing: **`loc` = labels, `iloc` = positions, and a boolean mask picks rows.**

---
# 2. Our dataset

A small, clean table so every output is easy to read.

In [3]:
df = pd.DataFrame({
    "student_id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "name":  ["Ravi", "Asha", "Karan", "Neha", "Amit", "Sara", "Dev", "Pooja", "Isha", "Rohan"],
    "city":  ["Indore", "Bhopal", "Delhi", "Indore", "Pune", "Bhopal", "Delhi", "Indore", "Pune", "Delhi"],
    "hours": [5.5, 7.0, 3.0, 6.5, 4.0, 8.0, 2.5, 6.0, 5.0, 3.5],
    "attendance": [80, 90, 65, 85, 70, 95, 60, 88, np.nan, 72],
    "score": [85, 92, 58, 88, 64, 97, 45, 81, 70, 52],
    "internet": ["Yes", "Yes", "No", "Yes", "No", "Yes", "No", "Yes", "Yes", "No"],
})
df

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.0,85,Yes
1,102,Asha,Bhopal,7.0,90.0,92,Yes
2,103,Karan,Delhi,3.0,65.0,58,No
3,104,Neha,Indore,6.5,85.0,88,Yes
4,105,Amit,Pune,4.0,70.0,64,No
5,106,Sara,Bhopal,8.0,95.0,97,Yes
6,107,Dev,Delhi,2.5,60.0,45,No
7,108,Pooja,Indore,6.0,88.0,81,Yes
8,109,Isha,Pune,5.0,NaN,70,Yes
9,110,Rohan,Delhi,3.5,72.0,52,No


---
# 3. Selecting columns

## 3.1 One column -> Series, several columns -> DataFrame

In [4]:
one = df["name"]
print(type(one))
one.head(3)

<class 'pandas.Series'>


0     Ravi
1     Asha
2    Karan
Name: name, dtype: str

In [5]:
two = df[["name", "score"]]        # note the DOUBLE brackets (a list inside [])
print(type(two))
two.head(3)

<class 'pandas.DataFrame'>


,name,score
0,Ravi,85
1,Asha,92
2,Karan,58


In [6]:
# Even a single column can come as a DataFrame if you use a list
print(type(df[["name"]]))

<class 'pandas.DataFrame'>


## 3.2 Dot notation `df.name` (works, but avoid in serious code)

`df.score` is a shortcut for `df["score"]`. It breaks when:
- the column name has a **space** (`"first name"`)
- the name clashes with a DataFrame method (`df.count`, `df.max`, `df.size` are **methods/attributes**, not your columns)
- you want to **create** a new column (`df.new_col = ...` does not create a column)

So prefer `df["score"]`.

In [7]:
print(df.score.mean())       # works here
print(df["score"].mean())    # always works

73.2
73.2


## 3.3 Select columns by type, by name pattern, by order

In [8]:
df.select_dtypes(include="number").head(3)

,student_id,hours,attendance,score
0,101,5.5,80.0,85
1,102,7.0,90.0,92
2,103,3.0,65.0,58


In [9]:
df.select_dtypes(exclude="number").head(3)

,name,city,internet
0,Ravi,Indore,Yes
1,Asha,Bhopal,Yes
2,Karan,Delhi,No


In [10]:
# filter(): pick columns by name pattern
print(df.filter(like="s").columns.tolist())             # name contains "s"
print(df.filter(regex="^s").columns.tolist())           # name starts with "s"
print(df.filter(items=["name", "city"]).columns.tolist())

['student_id', 'hours', 'score']
['student_id', 'score']
['name', 'city']


In [11]:
# Reorder columns: just select them in the order you want
df[["name", "score", "city", "hours"]].head(3)

,name,score,city,hours
0,Ravi,85,Indore,5.5
1,Asha,92,Bhopal,7.0
2,Karan,58,Delhi,3.0


In [12]:
# All columns except some
df.drop(columns=["student_id", "internet"]).head(3)

,name,city,hours,attendance,score
0,Ravi,Indore,5.5,80.0,85
1,Asha,Bhopal,7.0,90.0,92
2,Karan,Delhi,3.0,65.0,58


In [13]:
# Using a list of columns stored in a variable (very common in ML)
features = ["hours", "attendance"]
target = "score"
print(df[features].head(3))
print(df[target].head(3))

   hours  attendance
0    5.5        80.0
1    7.0        90.0
2    3.0        65.0
0    85
1    92
2    58
Name: score, dtype: int64


---
# 4. Selecting rows: the `[]` slicing quirk

`df[a:b]` with **slice** selects **rows by position** (end excluded, like Python lists).
But `df["col"]` selects a **column**. Same brackets, different meaning. This confuses beginners, so prefer `loc` / `iloc` for rows.

In [14]:
df[0:3]        # rows 0, 1, 2 (position 3 excluded)

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.0,85,Yes
1,102,Asha,Bhopal,7.0,90.0,92,Yes
2,103,Karan,Delhi,3.0,65.0,58,No


In [15]:
df[-2:]        # last 2 rows

,student_id,name,city,hours,attendance,score,internet
8,109,Isha,Pune,5.0,NaN,70,Yes
9,110,Rohan,Delhi,3.5,72.0,52,No


---
# 5. `loc` (by label)

Syntax: `df.loc[row_selection, column_selection]`

Rules of `loc`:
- uses **labels** (index values and column names)
- slices are **inclusive on both ends**
- accepts a single label, list of labels, slice, or boolean mask

In [16]:
df.loc[2]                  # one row (label 2) -> returns a Series

student_id      103
name          Karan
city          Delhi
hours           3.0
attendance     65.0
score            58
internet         No
Name: 2, dtype: object

In [17]:
df.loc[[0, 3, 5]]          # list of row labels -> DataFrame

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.0,85,Yes
3,104,Neha,Indore,6.5,85.0,88,Yes
5,106,Sara,Bhopal,8.0,95.0,97,Yes


In [18]:
df.loc[2:5]                # slice: rows 2,3,4,5  (5 IS included)

,student_id,name,city,hours,attendance,score,internet
2,103,Karan,Delhi,3.0,65.0,58,No
3,104,Neha,Indore,6.5,85.0,88,Yes
4,105,Amit,Pune,4.0,70.0,64,No
5,106,Sara,Bhopal,8.0,95.0,97,Yes


In [19]:
df.loc[2, "name"]          # one value

'Karan'

In [20]:
df.loc[2:5, ["name", "score"]]       # rows 2..5, two columns

,name,score
2,Karan,58
3,Neha,88
4,Amit,64
5,Sara,97


In [21]:
df.loc[2:5, "name":"hours"]          # column slice by name (also inclusive)

,name,city,hours
2,Karan,Delhi,3.0
3,Neha,Indore,6.5
4,Amit,Pune,4.0
5,Sara,Bhopal,8.0


In [22]:
df.loc[:, ["name", "score"]]         # ":" means ALL rows

,name,score
0,Ravi,85
1,Asha,92
2,Karan,58
3,Neha,88
4,Amit,64
5,Sara,97
6,Dev,45
7,Pooja,81
8,Isha,70
9,Rohan,52


In [23]:
df.loc[3:, :]                        # from row 3 to the end, all columns

,student_id,name,city,hours,attendance,score,internet
3,104,Neha,Indore,6.5,85.0,88,Yes
4,105,Amit,Pune,4.0,70.0,64,No
5,106,Sara,Bhopal,8.0,95.0,97,Yes
6,107,Dev,Delhi,2.5,60.0,45,No
7,108,Pooja,Indore,6.0,88.0,81,Yes
8,109,Isha,Pune,5.0,NaN,70,Yes
9,110,Rohan,Delhi,3.5,72.0,52,No


---
# 6. `iloc` (by position)

Syntax: `df.iloc[row_positions, column_positions]`

Rules of `iloc`:
- uses **integer positions** (0, 1, 2, ...), never labels
- slices **exclude the end** (like Python)
- supports negative positions

In [24]:
df.iloc[0]                 # first row (Series)

student_id       101
name            Ravi
city          Indore
hours            5.5
attendance      80.0
score             85
internet         Yes
Name: 0, dtype: object

In [25]:
df.iloc[-1]                # last row

student_id      110
name          Rohan
city          Delhi
hours           3.5
attendance     72.0
score            52
internet         No
Name: 9, dtype: object

In [26]:
df.iloc[0:3]               # rows at position 0, 1, 2

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.0,85,Yes
1,102,Asha,Bhopal,7.0,90.0,92,Yes
2,103,Karan,Delhi,3.0,65.0,58,No


In [27]:
df.iloc[[0, 2, 4], [1, 5]]            # specific rows and columns by position

,name,score
0,Ravi,85
2,Karan,58
4,Amit,64


In [28]:
df.iloc[0:4, 1:4]                     # first 4 rows, columns at position 1,2,3

,name,city,hours
0,Ravi,Indore,5.5
1,Asha,Bhopal,7.0
2,Karan,Delhi,3.0
3,Neha,Indore,6.5


In [29]:
df.iloc[::2]                          # every 2nd row (step)

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.0,85,Yes
2,103,Karan,Delhi,3.0,65.0,58,No
4,105,Amit,Pune,4.0,70.0,64,No
6,107,Dev,Delhi,2.5,60.0,45,No
8,109,Isha,Pune,5.0,NaN,70,Yes


In [30]:
df.iloc[3, 5]                         # one value: row position 3, column position 5

np.int64(88)

In [31]:
df.iloc[:, -2:]                       # last two columns

,score,internet
0,85,Yes
1,92,Yes
2,58,No
3,88,Yes
4,64,No
5,97,Yes
6,45,No
7,81,Yes
8,70,Yes
9,52,No


---
# 7. `loc` vs `iloc` (and the index trap)

| | `loc` | `iloc` |
|---|---|---|
| Uses | labels | positions |
| Slice end | **included** | **excluded** |
| Example | `df.loc[2:5]` -> 4 rows | `df.iloc[2:5]` -> 3 rows |
| Boolean mask | Yes (use this one) | Only with a plain array, not a Series |

### The trap: after sorting or filtering, labels no longer equal positions

In [32]:
by_score = df.sort_values("score", ascending=False)
by_score[["name", "score"]].head(4)

,name,score
5,Sara,97
1,Asha,92
3,Neha,88
0,Ravi,85


In [33]:
print("loc[0] :", by_score.loc[0, "name"])      # label 0 = Ravi
print("iloc[0]:", by_score.iloc[0]["name"])     # first row on screen = Sara (top score)

loc[0] : Ravi
iloc[0]: Sara


Same number, **different rows**. After sorting, the index labels travel with the rows, but positions restart from the top.

- Want "the row whose label is 0"? -> `loc[0]`
- Want "the first row I see"? -> `iloc[0]`
- Often you want a fresh 0..n-1 index: `by_score.reset_index(drop=True)`

In [34]:
by_score.reset_index(drop=True).loc[0, "name"]

'Sara'

---
# 8. Boolean filtering

The most powerful selection: pick rows by a **condition**.

How it works in 2 steps:
1. A condition creates a **boolean mask** (True/False for every row)
2. Passing the mask selects only the `True` rows

In [35]:
mask = df["score"] > 80
print(mask)

0     True
1     True
2    False
3     True
4    False
5     True
6    False
7     True
8    False
9    False
Name: score, dtype: bool


In [36]:
df[mask]

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.0,85,Yes
1,102,Asha,Bhopal,7.0,90.0,92,Yes
3,104,Neha,Indore,6.5,85.0,88,Yes
5,106,Sara,Bhopal,8.0,95.0,97,Yes
7,108,Pooja,Indore,6.0,88.0,81,Yes


In [37]:
# usually written in one line
df[df["score"] > 80]

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.0,85,Yes
1,102,Asha,Bhopal,7.0,90.0,92,Yes
3,104,Neha,Indore,6.5,85.0,88,Yes
5,106,Sara,Bhopal,8.0,95.0,97,Yes
7,108,Pooja,Indore,6.0,88.0,81,Yes


In [38]:
# Filter rows AND choose columns at the same time -> use loc
df.loc[df["score"] > 80, ["name", "score"]]

,name,score
0,Ravi,85
1,Asha,92
3,Neha,88
5,Sara,97
7,Pooja,81


## Combining conditions

| Operator | Meaning | Python word (do NOT use) |
|---|---|---|
| `&` | and | `and` |
| `|` | or | `or` |
| `~` | not | `not` |

**Always put each condition inside brackets.**

In [39]:
# AND: Indore students with score above 80
df[(df["city"] == "Indore") & (df["score"] > 80)]

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.0,85,Yes
3,104,Neha,Indore,6.5,85.0,88,Yes
7,108,Pooja,Indore,6.0,88.0,81,Yes


In [40]:
# OR: very high scorers or very low scorers
df[(df["score"] > 90) | (df["score"] < 50)]

,student_id,name,city,hours,attendance,score,internet
1,102,Asha,Bhopal,7.0,90.0,92,Yes
5,106,Sara,Bhopal,8.0,95.0,97,Yes
6,107,Dev,Delhi,2.5,60.0,45,No


In [41]:
# NOT: everyone except Delhi
df[~(df["city"] == "Delhi")]

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.0,85,Yes
1,102,Asha,Bhopal,7.0,90.0,92,Yes
3,104,Neha,Indore,6.5,85.0,88,Yes
4,105,Amit,Pune,4.0,70.0,64,No
5,106,Sara,Bhopal,8.0,95.0,97,Yes
7,108,Pooja,Indore,6.0,88.0,81,Yes
8,109,Isha,Pune,5.0,NaN,70,Yes


In [42]:
# Same as above using !=
df[df["city"] != "Delhi"]

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.0,85,Yes
1,102,Asha,Bhopal,7.0,90.0,92,Yes
3,104,Neha,Indore,6.5,85.0,88,Yes
4,105,Amit,Pune,4.0,70.0,64,No
5,106,Sara,Bhopal,8.0,95.0,97,Yes
7,108,Pooja,Indore,6.0,88.0,81,Yes
8,109,Isha,Pune,5.0,NaN,70,Yes


In [43]:
# Three conditions
df[(df["internet"] == "Yes") & (df["hours"] >= 5) & (df["score"] >= 85)]

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.0,85,Yes
1,102,Asha,Bhopal,7.0,90.0,92,Yes
3,104,Neha,Indore,6.5,85.0,88,Yes
5,106,Sara,Bhopal,8.0,95.0,97,Yes


## The two classic errors

In [44]:
# Error 1: forgetting brackets (& is stronger than > and <)
try:
    df[df["score"] > 80 & df["hours"] > 5]
except Exception as e:
    print("Error:", type(e).__name__)

Error: TypeError


In [45]:
# Error 2: using Python "and" instead of "&"
try:
    df[(df["score"] > 80) and (df["hours"] > 5)]
except Exception as e:
    print("Error:", type(e).__name__, "-", str(e)[:70])

Error: ValueError - The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.ite


In [46]:
# Save a mask in a variable when a condition is reused or long (more readable)
high_score = df["score"] >= 80
has_internet = df["internet"] == "Yes"

print("how many rows:", (high_score & has_internet).sum())      # True counts as 1
print("percentage   :", (high_score & has_internet).mean() * 100)

how many rows: 5
percentage   : 50.0


Handy trick: `mask.sum()` = number of matching rows, `mask.mean()` = fraction of matching rows.

---
# 9. `isin`, `between`, string filters, missing-value filters

## 9.1 `isin()`: value is in a list

In [47]:
df[df["city"].isin(["Indore", "Pune"])]

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.0,85,Yes
3,104,Neha,Indore,6.5,85.0,88,Yes
4,105,Amit,Pune,4.0,70.0,64,No
7,108,Pooja,Indore,6.0,88.0,81,Yes
8,109,Isha,Pune,5.0,NaN,70,Yes


In [48]:
# NOT in the list
df[~df["city"].isin(["Indore", "Pune"])]

,student_id,name,city,hours,attendance,score,internet
1,102,Asha,Bhopal,7.0,90.0,92,Yes
2,103,Karan,Delhi,3.0,65.0,58,No
5,106,Sara,Bhopal,8.0,95.0,97,Yes
6,107,Dev,Delhi,2.5,60.0,45,No
9,110,Rohan,Delhi,3.5,72.0,52,No


`isin` replaces long chains like `(city == "Indore") | (city == "Pune") | ...`.

## 9.2 `between()`: value in a range (both ends included)

In [49]:
df[df["score"].between(60, 85)]

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.0,85,Yes
4,105,Amit,Pune,4.0,70.0,64,No
7,108,Pooja,Indore,6.0,88.0,81,Yes
8,109,Isha,Pune,5.0,NaN,70,Yes


In [50]:
df[df["hours"].between(4, 6, inclusive="left")]     # 4 <= hours < 6

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.0,85,Yes
4,105,Amit,Pune,4.0,70.0,64,No
8,109,Isha,Pune,5.0,NaN,70,Yes


## 9.3 String filters with `.str`

In [51]:
df[df["name"].str.startswith("A")]

,student_id,name,city,hours,attendance,score,internet
1,102,Asha,Bhopal,7.0,90.0,92,Yes
4,105,Amit,Pune,4.0,70.0,64,No


In [52]:
df[df["name"].str.contains("a", case=False, na=False)]    # contains letter a/A

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.0,85,Yes
1,102,Asha,Bhopal,7.0,90.0,92,Yes
2,103,Karan,Delhi,3.0,65.0,58,No
3,104,Neha,Indore,6.5,85.0,88,Yes
4,105,Amit,Pune,4.0,70.0,64,No
5,106,Sara,Bhopal,8.0,95.0,97,Yes
7,108,Pooja,Indore,6.0,88.0,81,Yes
8,109,Isha,Pune,5.0,NaN,70,Yes
9,110,Rohan,Delhi,3.5,72.0,52,No


In [53]:
df[df["name"].str.len() > 4]

,student_id,name,city,hours,attendance,score,internet
2,103,Karan,Delhi,3.0,65.0,58,No
7,108,Pooja,Indore,6.0,88.0,81,Yes
9,110,Rohan,Delhi,3.5,72.0,52,No


- `case=False` ignores upper/lower case
- `na=False` treats missing text as "no match" (avoids errors with NaN)
- Full text cleaning (`strip`, `lower`, `replace`) comes in chapter 08.

## 9.4 Missing-value filters

In [54]:
df[df["attendance"].isna()]           # rows where attendance is missing

,student_id,name,city,hours,attendance,score,internet
8,109,Isha,Pune,5.0,NaN,70,Yes


In [55]:
df[df["attendance"].notna()].shape    # rows where it is present

(9, 7)

In [56]:
# Rows with NO missing value at all / with ANY missing value
print(df.dropna().shape)
print(df[df.isna().any(axis=1)].shape)

(9, 7)
(1, 7)


> Never write `df["col"] == np.nan`. NaN is not equal to anything, even itself, so it always gives False. Use `isna()`.

---
# 10. `query()`

`query()` lets you write the condition as a **string**. Often shorter and easier to read.

In [57]:
df.query("score > 80")

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.0,85,Yes
1,102,Asha,Bhopal,7.0,90.0,92,Yes
3,104,Neha,Indore,6.5,85.0,88,Yes
5,106,Sara,Bhopal,8.0,95.0,97,Yes
7,108,Pooja,Indore,6.0,88.0,81,Yes


In [58]:
df.query("city == 'Indore' and score > 80")        # use and / or / not inside query

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.0,85,Yes
3,104,Neha,Indore,6.5,85.0,88,Yes
7,108,Pooja,Indore,6.0,88.0,81,Yes


In [59]:
df.query("city in ['Indore', 'Pune'] and hours >= 5")

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.0,85,Yes
3,104,Neha,Indore,6.5,85.0,88,Yes
7,108,Pooja,Indore,6.0,88.0,81,Yes
8,109,Isha,Pune,5.0,NaN,70,Yes


In [60]:
# Use Python variables with @
min_score = 85
cities = ["Bhopal", "Delhi"]

print(df.query("score >= @min_score"))
print()
print(df.query("city in @cities"))

   student_id  name    city  hours  attendance  score internet
0         101  Ravi  Indore    5.5        80.0     85      Yes
1         102  Asha  Bhopal    7.0        90.0     92      Yes
3         104  Neha  Indore    6.5        85.0     88      Yes
5         106  Sara  Bhopal    8.0        95.0     97      Yes

   student_id   name    city  hours  attendance  score internet
1         102   Asha  Bhopal    7.0        90.0     92      Yes
2         103  Karan   Delhi    3.0        65.0     58       No
5         106   Sara  Bhopal    8.0        95.0     97      Yes
6         107    Dev   Delhi    2.5        60.0     45       No
9         110  Rohan   Delhi    3.5        72.0     52       No


In [61]:
# column names with spaces need backticks
tmp = df.rename(columns={"hours": "study hours"})
tmp.query("`study hours` > 6")

,student_id,name,city,study hours,attendance,score,internet
1,102,Asha,Bhopal,7.0,90.0,92,Yes
3,104,Neha,Indore,6.5,85.0,88,Yes
5,106,Sara,Bhopal,8.0,95.0,97,Yes


| | Boolean mask | `query()` |
|---|---|---|
| Style | `df[(a) & (b)]` | `df.query("a and b")` |
| Needs `df[...]` repeated | Yes | No |
| Works with variables | Directly | Use `@var` |
| Good for | Any condition, complex logic | Readable multi-condition filters |

Both give the same result. Interviewers expect you to know the boolean mask version well.

---
# 11. `at`, `iat`, `where`, `np.where`

## 11.1 `at` and `iat`: single value, fast

In [62]:
print(df.at[3, "name"])       # by label
print(df.iat[3, 1])           # by position

Neha
Neha


## 11.2 `nlargest` / `nsmallest` / `head` as selection

In [63]:
df.nlargest(3, "score")[["name", "score"]]

,name,score
5,Sara,97
1,Asha,92
3,Neha,88


In [64]:
df.nsmallest(2, "hours")[["name", "hours"]]

,name,hours
6,Dev,2.5
2,Karan,3.0


## 11.3 `where()`: keep values where True, replace the rest

In [65]:
s = df["score"]
print(s.where(s >= 60).head(6).tolist())        # below 60 becomes NaN
print(s.where(s >= 60, 60).head(6).tolist())    # below 60 replaced by 60 (a floor)

[85.0, 92.0, nan, 88.0, 64.0, 97.0]
[85, 92, 60, 88, 64, 97]


## 11.4 `np.where()`: create a new column from a condition

Pattern: `np.where(condition, value_if_true, value_if_false)`. Very common in feature engineering.

In [66]:
tmp = df.copy()
tmp["result"] = np.where(tmp["score"] >= 60, "Pass", "Fail")
tmp[["name", "score", "result"]]

,name,score,result
0,Ravi,85,Pass
1,Asha,92,Pass
2,Karan,58,Fail
3,Neha,88,Pass
4,Amit,64,Pass
5,Sara,97,Pass
6,Dev,45,Fail
7,Pooja,81,Pass
8,Isha,70,Pass
9,Rohan,52,Fail


In [67]:
# More than two outcomes: np.select
conditions = [tmp["score"] >= 85, tmp["score"] >= 60]
choices = ["A", "B"]
tmp["grade"] = np.select(conditions, choices, default="C")
tmp[["name", "score", "grade"]]

,name,score,grade
0,Ravi,85,A
1,Asha,92,A
2,Karan,58,C
3,Neha,88,A
4,Amit,64,B
5,Sara,97,A
6,Dev,45,C
7,Pooja,81,B
8,Isha,70,B
9,Rohan,52,C


---
# 12. Setting values safely (and Copy-on-Write)

To **change** values, use `loc[rows, column] = value` in **one step**.

In [68]:
work = df.copy()                       # we work on a copy to keep df safe

work.loc[work["score"] < 50, "internet"] = "Check"
work.loc[work["attendance"].isna(), "attendance"] = work["attendance"].mean()
work.loc[work["city"] == "Delhi", ["hours", "score"]] = [0, 0]

work.head(10)

,student_id,name,city,hours,attendance,score,internet
0,101,Ravi,Indore,5.5,80.000000,85,Yes
1,102,Asha,Bhopal,7.0,90.000000,92,Yes
2,103,Karan,Delhi,0.0,65.000000,0,No
3,104,Neha,Indore,6.5,85.000000,88,Yes
4,105,Amit,Pune,4.0,70.000000,64,No
5,106,Sara,Bhopal,8.0,95.000000,97,Yes
6,107,Dev,Delhi,0.0,60.000000,0,Check
7,108,Pooja,Indore,6.0,88.000000,81,Yes
8,109,Isha,Pune,5.0,78.333333,70,Yes
9,110,Rohan,Delhi,0.0,72.000000,0,No


## The chained assignment trap

`df[condition]["col"] = value` is **two steps**: first a filtered copy is made, then the value is set **on that temporary copy**. The original DataFrame is not changed.

In pandas 3 this is officially not allowed (Copy-on-Write), and pandas warns you:

In [69]:
wrong = df.copy()

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    wrong[wrong["score"] < 50]["internet"] = "Check"      # WRONG way
    print("Warning:", w[0].category.__name__ if w else "none")

print("Rows changed:", (wrong["internet"] == "Check").sum())    # 0 -> nothing happened

Rows changed: 0


In [70]:
right = df.copy()
right.loc[right["score"] < 50, "internet"] = "Check"          # RIGHT way
print("Rows changed:", (right["internet"] == "Check").sum())

Rows changed: 1


## Copy-on-Write: selections are independent copies

In pandas 3, a DataFrame you get from selecting never changes the original behind your back.

In [71]:
sub = df[["name", "score"]]            # a new object
sub.loc[0, "score"] = 0                # change the sub-table

print("sub score :", sub.loc[0, "score"])
print("df  score :", df.loc[0, "score"])      # original unchanged

sub score : 0
df  score : 85


So the rule is simple: **to modify the original, assign back to it** (`df.loc[...] = ...` or `df = df.something()`).

---
# 13. Selecting with a custom index

If a column is a natural key (like `name` or `student_id`), make it the index and select rows by that key.

In [72]:
by_name = df.set_index("name")
by_name.head(3)

,student_id,city,hours,attendance,score,internet
name,,,,,,
Ravi,101,Indore,5.5,80.0,85,Yes
Asha,102,Bhopal,7.0,90.0,92,Yes
Karan,103,Delhi,3.0,65.0,58,No


In [73]:
by_name.loc["Neha"]

student_id       104
city          Indore
hours            6.5
attendance      85.0
score             88
internet         Yes
Name: Neha, dtype: object

In [74]:
by_name.loc[["Ravi", "Sara"], ["city", "score"]]

,city,score
name,,
Ravi,Indore,85
Sara,Bhopal,97


In [75]:
by_name.loc["Asha":"Amit"]          # label slice follows the ORDER in the table, end included

,student_id,city,hours,attendance,score,internet
name,,,,,,
Asha,102,Bhopal,7.0,90.0,92,Yes
Karan,103,Delhi,3.0,65.0,58,No
Neha,104,Indore,6.5,85.0,88,Yes
Amit,105,Pune,4.0,70.0,64,No


In [76]:
by_name.reset_index().head(3)       # move the index back to a normal column

,name,student_id,city,hours,attendance,score,internet
0,Ravi,101,Indore,5.5,80.0,85,Yes
1,Asha,102,Bhopal,7.0,90.0,92,Yes
2,Karan,103,Delhi,3.0,65.0,58,No


Notes:
- A missing label raises `KeyError`. Use `by_name.index.isin([...])` or `df.reindex([...])` if labels may be missing.
- Index labels do not have to be unique, but unique labels make `loc` simple and safe.

In [77]:
try:
    by_name.loc["Nobody"]
except KeyError as e:
    print("KeyError:", e)

KeyError: 'Nobody'


---
# 14. Mini AI/ML example: X, y and train/test split

Every ML project does this with selection:
1. Pick **feature columns** -> `X`
2. Pick the **target column** -> `y`
3. Split rows into **train** and **test**

In [78]:
rng = np.random.default_rng(7)
n = 200

ml = pd.DataFrame({
    "hours": rng.normal(5, 1.5, n).clip(0.5, 10).round(1),
    "attendance": rng.normal(78, 10, n).clip(40, 100).round(0),
    "prev_score": rng.normal(62, 12, n).clip(15, 100).round(0),
    "internet": rng.choice([0, 1], n, p=[0.2, 0.8]),
})
ml["score"] = (15 + 5 * ml["hours"] + 0.25 * ml["attendance"] + 0.2 * ml["prev_score"]
               + rng.normal(0, 5, n)).clip(0, 100).round(0)
ml["passed"] = (ml["score"] >= 60).astype(int)

ml.head()

,hours,attendance,prev_score,internet,score,passed
0,5.0,66.0,64.0,1,71.0,1
1,5.4,87.0,67.0,1,74.0,1
2,4.6,83.0,57.0,0,60.0,1
3,3.7,87.0,74.0,1,74.0,1
4,4.3,97.0,65.0,1,73.0,1


In [79]:
# 1 and 2: features and target
features = ["hours", "attendance", "prev_score", "internet"]
X = ml[features]            # DataFrame (2D)
y = ml["passed"]            # Series (1D)
print(X.shape, y.shape)

(200, 4) (200,)


In [80]:
# 3: shuffle all rows, then split 80% / 20% using iloc
shuffled = ml.sample(frac=1, random_state=42).reset_index(drop=True)

split = int(0.8 * len(shuffled))
train = shuffled.iloc[:split]
test = shuffled.iloc[split:]

print("train:", train.shape, "| test:", test.shape)

train: (160, 6) | test: (40, 6)


In [81]:
X_train, y_train = train[features], train["passed"]
X_test, y_test = test[features], test["passed"]

# check that the pass-rate is similar in both parts
print("pass rate train:", round(y_train.mean(), 3))
print("pass rate test :", round(y_test.mean(), 3))

pass rate train: 0.9
pass rate test : 0.85


In [82]:
# Selecting a subset to analyse: weak students with internet
weak_online = ml[(ml["passed"] == 0) & (ml["internet"] == 1)]
print(len(weak_online), "students")
weak_online[features].describe().loc[["mean", "min", "max"]].round(1)

16 students


,hours,attendance,prev_score,internet
mean,3.0,72.5,55.2,1.0
min,1.2,59.0,42.0,1.0
max,4.6,83.0,72.0,1.0


`sklearn.model_selection.train_test_split` does the shuffle-and-split for you (with `stratify=y` to keep class ratios). We did it by hand so you understand exactly what `iloc` and `sample` do underneath.

---
# 15. Pandas 3.x notes

- **Copy-on-Write is on:** results of selections behave as independent copies. Chained assignment (`df[cond]["col"] = x`) does not work and warns. Always use `df.loc[cond, "col"] = x`.
- `df[0]` (integer in brackets) looks for a **column named 0**. It does not give the first row. Use `iloc[0]`.
- Text columns are `str` dtype. `.str` methods work as before; keep `na=False` in `contains` when data may have NaN.
- Prefer `loc` / `iloc` over plain `[]` for rows, to avoid ambiguity.

---
# 16. Common mistakes

| Mistake | Fix |
|---|---|
| `df["a", "b"]` | `df[["a", "b"]]` |
| `df[df.a > 1 and df.b < 5]` | `df[(df.a > 1) & (df.b < 5)]` |
| `df[df["a"] > 1 & df["b"] < 5]` (no brackets) | Wrap every condition in `( )` |
| `df.loc[2:5]` expecting 3 rows | `loc` includes the end -> 4 rows |
| Using `loc[0]` after sorting, expecting the top row | Use `iloc[0]` or `reset_index(drop=True)` |
| `df[cond]["col"] = value` | `df.loc[cond, "col"] = value` |
| `df["col"] == np.nan` | `df["col"].isna()` |
| `df[0]` to get first row | `df.iloc[0]` |
| Many `or` conditions on one column | `isin([...])` |
| `str.contains` on missing text gives errors | `str.contains("x", na=False)` |
| Forgetting that `df[["a"]]` is a DataFrame, `df["a"]` a Series | Check `type()` when shapes surprise you |
| Looping row by row to filter | Use a boolean mask |

---
# 17. Interview questions

**Q1. Difference between `loc` and `iloc`?**
`loc` selects by labels and includes the slice end. `iloc` selects by integer position and excludes the slice end.

**Q2. How do you filter rows with multiple conditions?**
Combine boolean masks with `&`, `|`, `~` and wrap each condition in brackets: `df[(df.a > 1) & (df.b == "x")]`.

**Q3. Why do we use `&` and not `and`?**
`and` needs a single True/False, but a Series holds many values, so pandas raises "truth value is ambiguous". `&` works element by element.

**Q4. What is chained assignment and why is it bad?**
`df[cond]["col"] = v` first creates a temporary copy and then assigns on that copy, so the original is unchanged. Use `df.loc[cond, "col"] = v`.

**Q5. `df["a"]` vs `df[["a"]]`?**
First returns a Series, second returns a one-column DataFrame.

**Q6. How do you select rows where a value is in a list?**
`df[df["col"].isin([...])]`. Negate with `~`.

**Q7. How do you select all columns except some?**
`df.drop(columns=[...])`, or `df.loc[:, ~df.columns.isin([...])]`.

**Q8. After `sort_values`, what does `df.loc[0]` return?**
The row whose index label is 0 (wherever it moved), not the first row. `iloc[0]` gives the first row.

**Q9. How do you split data into train and test using only pandas?**
Shuffle with `df.sample(frac=1, random_state=...)`, then take `iloc[:split]` and `iloc[split:]`. In practice use `train_test_split`.

**Q10. How do you count rows matching a condition?**
`(df["col"] > x).sum()`; the fraction is `(df["col"] > x).mean()`.

---
# 18. Practice questions

Run this cell first to get the practice data.

In [83]:
emp = pd.DataFrame({
    "emp_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    "name":   ["Amit", "Bina", "Chirag", "Divya", "Esha", "Farhan", "Gita", "Harsh", "Isha", "Jay", "Kiran", "Lata"],
    "dept":   ["ML", "Web", "ML", "Data", "Web", "ML", "Data", "Web", "ML", "Data", "Web", "ML"],
    "exp":    [2, 5, 3, 1, 8, 4, 6, 2, 7, 3, 9, 1],
    "salary": [12, 18, 15, 9, 25, 16, 20, 11, 22, 13, 27, np.nan],     # in LPA
    "remote": [True, False, True, False, True, True, False, False, True, False, True, True],
})
emp

,emp_id,name,dept,exp,salary,remote
0,1,Amit,ML,2,12.0,True
1,2,Bina,Web,5,18.0,False
2,3,Chirag,ML,3,15.0,True
3,4,Divya,Data,1,9.0,False
4,5,Esha,Web,8,25.0,True
5,6,Farhan,ML,4,16.0,True
6,7,Gita,Data,6,20.0,False
7,8,Harsh,Web,2,11.0,False
8,9,Isha,ML,7,22.0,True
9,10,Jay,Data,3,13.0,False


**Q1.** Select only the `name` and `salary` columns.
**Q2.** Select rows at positions 2 to 5 (4 rows) with `iloc`, then the same 4 rows with `loc`.
**Q3.** Select the last 3 rows and the last 2 columns.
**Q4.** Employees from the `ML` department with salary above 15.
**Q5.** Employees from `Web` or `Data` with experience between 3 and 8 (both included).
**Q6.** Employees whose name starts with a vowel (A, E, I, O, U).
**Q7.** Employees whose salary is missing.
**Q8.** How many remote employees are there, and what percentage of all employees is that?
**Q9.** Using `query()`, find employees with `exp >= 5` that are not in the `ML` department.
**Q10.** Set the missing salary to the **median salary of the ML department** (use `loc`, on a copy).
**Q11.** Sort by salary (high to low). What does `loc[0]` return vs `iloc[0]`?
**Q12.** Top 3 highest paid employees: show `name`, `dept`, `salary`.

### Solutions

In [84]:
# Q1
emp[["name", "salary"]]

,name,salary
0,Amit,12.0
1,Bina,18.0
2,Chirag,15.0
3,Divya,9.0
4,Esha,25.0
5,Farhan,16.0
6,Gita,20.0
7,Harsh,11.0
8,Isha,22.0
9,Jay,13.0


In [85]:
# Q2
print(emp.iloc[2:6])
print(emp.loc[2:5])

   emp_id    name  dept  exp  salary  remote
2       3  Chirag    ML    3    15.0    True
3       4   Divya  Data    1     9.0   False
4       5    Esha   Web    8    25.0    True
5       6  Farhan    ML    4    16.0    True
   emp_id    name  dept  exp  salary  remote
2       3  Chirag    ML    3    15.0    True
3       4   Divya  Data    1     9.0   False
4       5    Esha   Web    8    25.0    True
5       6  Farhan    ML    4    16.0    True


In [86]:
# Q3
emp.iloc[-3:, -2:]

,salary,remote
9,13.0,False
10,27.0,True
11,NaN,True


In [87]:
# Q4
emp[(emp["dept"] == "ML") & (emp["salary"] > 15)]

,emp_id,name,dept,exp,salary,remote
5,6,Farhan,ML,4,16.0,True
8,9,Isha,ML,7,22.0,True


In [88]:
# Q5
emp[emp["dept"].isin(["Web", "Data"]) & emp["exp"].between(3, 8)]

,emp_id,name,dept,exp,salary,remote
1,2,Bina,Web,5,18.0,False
4,5,Esha,Web,8,25.0,True
6,7,Gita,Data,6,20.0,False
9,10,Jay,Data,3,13.0,False


In [89]:
# Q6
emp[emp["name"].str[0].isin(list("AEIOU"))]
# alternative: emp[emp["name"].str.match("^[AEIOU]")]

,emp_id,name,dept,exp,salary,remote
0,1,Amit,ML,2,12.0,True
4,5,Esha,Web,8,25.0,True
8,9,Isha,ML,7,22.0,True


In [90]:
# Q7
emp[emp["salary"].isna()]

,emp_id,name,dept,exp,salary,remote
11,12,Lata,ML,1,NaN,True


In [91]:
# Q8
count = emp["remote"].sum()
pct = emp["remote"].mean() * 100
print(count, "remote employees =", round(pct, 1), "%")

7 remote employees = 58.3 %


In [92]:
# Q9
emp.query("exp >= 5 and dept != 'ML'")

,emp_id,name,dept,exp,salary,remote
1,2,Bina,Web,5,18.0,False
4,5,Esha,Web,8,25.0,True
6,7,Gita,Data,6,20.0,False
10,11,Kiran,Web,9,27.0,True


In [93]:
# Q10
fixed = emp.copy()
ml_median = fixed.loc[fixed["dept"] == "ML", "salary"].median()
fixed.loc[fixed["salary"].isna(), "salary"] = ml_median
print("ML median:", ml_median)
fixed.tail(3)

ML median: 15.5


,emp_id,name,dept,exp,salary,remote
9,10,Jay,Data,3,13.0,False
10,11,Kiran,Web,9,27.0,True
11,12,Lata,ML,1,15.5,True


In [94]:
# Q11
srt = emp.sort_values("salary", ascending=False)
print("loc[0] :", srt.loc[0, "name"], "(label 0 is Amit)")
print("iloc[0]:", srt.iloc[0]["name"], "(top salary row)")

loc[0] : Amit (label 0 is Amit)
iloc[0]: Kiran (top salary row)


In [95]:
# Q12
emp.nlargest(3, "salary")[["name", "dept", "salary"]]

,name,dept,salary
10,Kiran,Web,27.0
4,Esha,Web,25.0
8,Isha,ML,22.0


---
# 19. Cheat sheet

```python
# Columns
df["a"]                     # Series
df[["a", "b"]]              # DataFrame
df.select_dtypes(include="number")
df.filter(like="s") / df.filter(regex="^s")
df.drop(columns=["a"])

# Rows / cells by label (end included)
df.loc[2]   df.loc[[0, 3]]   df.loc[2:5]   df.loc[2:5, "a":"c"]   df.loc[:, ["a", "b"]]

# Rows / cells by position (end excluded)
df.iloc[0]   df.iloc[-1]   df.iloc[0:3]   df.iloc[::2]   df.iloc[[0, 2], [1, 3]]

# Boolean filtering
df[df["a"] > 5]
df[(df["a"] > 5) & (df["b"] == "x")]      # & | ~  with brackets
df.loc[cond, ["a", "b"]]                   # filter rows + choose columns
df["a"].isin([...])   df["a"].between(1, 5)
df["t"].str.contains("x", case=False, na=False)   df["t"].str.startswith("A")
df["a"].isna()   df["a"].notna()
df.query("a > 5 and b == 'x'")   df.query("a > @var")

# Single value
df.at[label, "col"]   df.iat[row, col]

# Conditional values
s.where(cond, other)   np.where(cond, x, y)   np.select(conds, choices, default)

# Set values (one step!)
df.loc[cond, "col"] = value

# Index tricks
df.set_index("col")   df.reset_index(drop=True)   df.sort_values("a")
df.nlargest(3, "a")   df.sample(frac=1, random_state=42)

# Counting
cond.sum()   cond.mean()
```

**Next chapter:** `06_adding_updating_dropping`: create new columns, update values, rename, drop, insert, and the `assign` method for clean pipelines.